# nb_mdf_10_landing_to_bronze - generic, metadata-driven ingestion into Bronze

One notebook for **every** source. It reads `meta.ingestion_object`, finds `load_date=` folders in Landing that are
newer than the object's Bronze watermark, reads them with the right reader (csv / txt / json / xlsx), adds audit
columns and writes them to `bronze.<table>`.

**Production behaviours shown**
* **Idempotent reruns** - each load date is written with `replaceWhere`, so re-running a day replaces it instead of duplicating it.
* **Schema drift** - new source columns are absorbed (`mergeSchema`) and reported in `meta.run_log`.
* **Per-object isolation** - one failing object is logged and does not stop the others; the notebook fails at the end so the pipeline sees it.
* **Two calling styles** - a pipeline ForEach passes one `object_ids`, or the notebook orchestrator passes `load_group = all`
  (one Spark session for many objects = far cheaper than one session per table).

In [ ]:
object_ids = ""          # comma-separated object_ids; empty = use load_group
load_group = "all"       # reference | transactional | telemetry | all
reprocess_dates = ""     # e.g. "2026-11-03" to force a rerun of specific load dates (ignores the watermark)
parent_run_id = ""       # pipeline run id / orchestrator run id for end-to-end traceability
fail_on_error = True

In [ ]:
%run nb_mdf_utils

In [ ]:
def ingest_to_bronze(obj, run_id):
    started = utcnow()
    oid = obj["object_id"]
    target = tname("bronze", obj["bronze_table"])
    try:
        available = landing_dates(obj)
        wm = get_watermark(oid, "bronze")
        if reprocess_dates.strip():
            dates = [d.strip() for d in reprocess_dates.split(",") if d.strip() in available]
        else:
            dates = [d for d in available if wm is None or d > wm]
        if not dates:
            log_run(run_id, parent_run_id, oid, "bronze", "SKIPPED", started, message="no new load_date folders")
            return "SKIPPED"

        df = read_landing(obj, dates)
        df = (df.withColumn("_source_system", F.lit(obj["source_system"]))
                .withColumn("_object_id", F.lit(oid))
                .withColumn("_ingested_at", F.current_timestamp())
                .withColumn("_batch_id", F.lit(run_id)))
        df = df.cache()
        rows = df.count()

        msg = []
        if table_exists(target):
            existing = {c.lower() for c in spark.table(target).columns}
            drift = [c for c in df.columns if c.lower() not in existing]
            if drift:
                msg.append(f"SCHEMA DRIFT - new source columns absorbed: {', '.join(drift)}")
            in_list = ", ".join(f"'{d}'" for d in dates)
            (df.write.format("delta").mode("overwrite").option("mergeSchema", "true")
               .option("replaceWhere", f"_load_date IN ({in_list})").saveAsTable(target))
        else:
            df.write.format("delta").saveAsTable(target)

        # Source watermark (used by the pipeline to build the next incremental source query)
        wm_col = obj.get("watermark_column")
        if obj["ingest_method"] == "copy_db" and wm_col and sanitize(wm_col) in df.columns:
            set_watermark(oid, "source", df.agg(F.max(F.col(sanitize(wm_col)))).first()[0], run_id)
        set_watermark(oid, "bronze", max(dates + ([wm] if wm else [])), run_id)   # never moves backwards on a reprocess
        msg.append(f"load dates {', '.join(dates)}")
        df.unpersist()
        log_run(run_id, parent_run_id, oid, "bronze", "SUCCESS", started, rows_read=rows, rows_written=rows,
                message="; ".join(msg))
        return "SUCCESS"
    except Exception as e:
        log_run(run_id, parent_run_id, oid, "bronze", "FAILED", started,
                message=f"{type(e).__name__}: {str(e)[:1500]}")
        traceback.print_exc()
        return "FAILED"

In [ ]:
run_id = new_run_id("bronze")
objects = get_objects(object_ids, load_group)
print(f"{len(objects)} object(s) to ingest - run {run_id}")
results = {o["object_id"]: ingest_to_bronze(o, run_id) for o in objects}
display(spark.table(tname("meta", "run_log")).where(F.col("run_id") == run_id)
        .select("object_id", "status", "rows_read", "duration_sec", "message"))

In [ ]:
failed = [k for k, v in results.items() if v == "FAILED"]
summary = json.dumps({"run_id": run_id, "results": results, "failed": failed})
if failed and fail_on_error:
    raise Exception(f"Bronze ingestion failed for: {failed}. See meta.run_log for run {run_id}")
notebookutils.notebook.exit(summary)